# Particle Swarm Optimization Algorithm for Feature Selection

In [1]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score

from matplotlib import pyplot as plt

from sklearn.ensemble import RandomForestClassifier

from random import randint

from sklearn.metrics import f1_score


In [40]:
#read input files
def readfiles(trainfile,testfile,valfile):
    print('reading files')
    traindf=pd.read_csv(trainfile)
    Xtrain=traindf.iloc[:,0:-1].values
    ytrain=traindf.iloc[:,-1].values

    testdf=pd.read_csv(testfile)
    Xtest=testdf.iloc[:,0:-1].values
    ytest=testdf.iloc[:,-1].values
    
    valdf=pd.read_csv(valfile)
    Xval=valdf.iloc[:,0:-1].values
    yval=valdf.iloc[:,-1].values
    
    featurelist=traindf.columns[0:-1]

    dimensions=Xtrain.shape[1]
    return Xtrain,Xtest,Xval,ytrain,ytest,yval,dimensions,featurelist

In [41]:
#to generate initial population
def generate_particles(n_particles,dimensions):
  
    """
    n_particles,dimensions,options
        
    """
    #create the particles. since this is a feature selection problem,
    #dimensions represent the number of features
    #the problem now becomes a binary one.
    #particles will have values made up of 0s and 1s
    #1 means the feature has been selected , 0 means feature is not selected
    position=np.random.randint(2,size=(n_particles,dimensions))  #generators.py file
    min_velocity, max_velocity=(0,1)

    #compute velocity
    velocity=(max_velocity - min_velocity) * np.random.random_sample(size=(n_particles,dimensions)) + min_velocity #generators.py file

    scores=np.array([-np.inf for i in range(n_particles)])
    pbest_position=position.copy()
   
    pbest=[[position[i],scores[i]] for i in range(n_particles)]
   
    
  
    return position,velocity,pbest

In [42]:
#compute position
def compute_position(velocity,dimensions):

    __sigmoid=lambda x: 1/(1+np.exp(-x))

    positions=(np.random.random_sample(size=dimensions)< __sigmoid(velocity)) * 1
    positions=positions.flatten()

        #print('SHAPE',positions.shape)

    #print(positions.flatten())
        
    return positions

In [44]:
def compute_velocity(current_position,pbest_position,gbest_position,current_velocity,options):

        # Prepare parameters
    swarm_size = current_position.shape
    c1 = options["c1"]
    c2 = options["c2"]
    w = options["w"]
        # Compute for cognitive and social terms
    cognitive = (c1* np.random.uniform(0, 1, swarm_size)
            * (pbest_position - current_position))
    social = (
            c2
            * np.random.uniform(0, 1, swarm_size)
            * (gbest_position  - current_position)
        )
        # Compute temp velocity (subject to clamping if possible)
    updated_velocity = (w * current_velocity) + cognitive + social
    
    return updated_velocity


In [45]:
#evaluate the chromosome/feature subset
def evaluate_particle(position,classifier,Xtrain,Xtest,ytrain,ytest):
 
    subset_train=Xtrain[:,position==1]
    subset_test=Xtest[:,position==1]
        
    classifier.fit(subset_train,ytrain)
    ypred=classifier.predict(subset_test)

    score=f1_score(ytest,ypred,average='weighted')
    return score


In [46]:
#get personal best
def get_pbest(p1,p2):
    if p1[1]>=p2[1]:
        return p1
    else:
        return p2

In [47]:
def get_gbest(scores):
    gbest_index=np.argsort(scores)[::-1][0]
    return gbest_index
    

In [54]:

#Main Algol to start and run the 

def PSO(trainfile,testfile,valfile,n_iters,pop_size,options):

    
    Xtrain,Xtest,Xval,ytrain,ytest,yval,dimensions,featurelist=readfiles(trainfile,testfile,valfile)
  
    position,velocity,pbest=generate_particles(pop_size,dimensions)
    
    classifier=RandomForestClassifier(random_state=20)


    gbest_records={}


    
    for iter_ in range(n_iters):              


        #evaluate particles/solutions/feature subsets
        scores=[evaluate_particle(position[i],classifier,Xtrain,Xval,ytrain,yval) for i in range(pop_size)]

   

        #find pbest
        pbest=[get_pbest([position[i],scores[i]],pbest[i]) for i in range(pop_size)]


        #find gbest
        gbest_index=get_gbest( [ pbest[i][1] for i in range(pop_size)     ]   )
        gbest_position, gbest_score=pbest[gbest_index]
        
     

        #record gbest
        gbest_records[iter_+1]={'features':featurelist[gbest_position==True], 'score':gbest_score}

        print('generation {} of {} : {}, {}'.format(iter_, n_iters,gbest_score,np.count_nonzero(gbest_position)))

        
        #compute velocity
        velocity=[compute_velocity(position[i],pbest[i][0], gbest_position,velocity[i],options) for i in range(pop_size)]
      

        #compute position
        position=[compute_position(velocity[i],dimensions) for i in range(pop_size)]
     

    #evaluate the identified optimal subset
    test_score=evaluate_particle(gbest_position,classifier,Xtrain,Xtest,ytrain,ytest)


    print('score of best subset on the test dataset:',test_score)

    return gbest_records,test_score
            

In [102]:
#input files
data_path="https://raw.githubusercontent.com/vappiah/Machine-Learning-Tutorials/refs/heads/main/datasets/Wisconsin"

trainfile='%s/train.csv'%data_path
testfile='%s/test.csv'%data_path
valfile='%s/val.csv'%data_path


#parameters
n_iters=10
pop_size=10
options={'c1':2,'c2':2,'w':0.9,'p':2}

#run algorithm
gbest_records,test_score=PSO(trainfile,testfile,valfile,n_iters,pop_size,options)